In [12]:
import sys
import os
from pathlib import Path
import pandas as pd
import requests
import io
from dotenv import load_dotenv
import openpyxl
from urllib.parse import quote

load_dotenv()
DIR = Path().resolve().parent
if str(DIR) not in sys.path:
    sys.path.append(str(DIR))

from src.performance_dashboard.utils import get_secret
from src.performance_dashboard.services.sharepoint_client import *


In [13]:
path1 = "opdrachtgever_1.parquet"
path2 = "peer_1.parquet"
sub_folder = get_secret("TRANSFORMED_RESPONSES_FOLDER")
processed_folder = "processed"

hr_folder = get_secret("HR_CYCLE_FOLDER")
combined_df_folder = get_secret("COMBINED_RESPONSES_FOLDER")

    # Pad opbouwen
file_path = f"{hr_folder}/{sub_folder}/{path1}" if sub_folder else f"{hr_folder}/{path1}"
encoded_path = quote(file_path, safe='/')

In [14]:
TENANT_ID = get_secret("SHAREPOINT_TENANT_ID")  
CLIENT_ID = get_secret("SHAREPOINT_CLIENT_ID")  
CLIENT_SECRET = get_secret("SHAREPOINT_VALUE")  
SITE_URL = get_secret("SHAREPOINT_SITE")  
HR_FOLDER = get_secret("HR_CYCLE_FOLDER")
SITE_NAME = 'VeneficusDataSafe'

In [15]:
auth_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
auth_data = {
    'client_id': CLIENT_ID,
    'scope': 'https://graph.microsoft.com/.default',
    'client_secret': CLIENT_SECRET,
    'grant_type': 'client_credentials'
}
token = requests.post(auth_url, data=auth_data).json().get('access_token')

# 2. Get Drive ID
headers = {'Authorization': f'Bearer {token}'}
site_url = f"https://graph.microsoft.com/v1.0/sites/veneficus.sharepoint.com:/sites/{SITE_NAME}:/drive"
response = requests.get(site_url, headers=headers).json()

print(f"Drive ID: {response.get('id')}")

Drive ID: b!qE1157W_e0K66S2qnQ6svNNXejuaHWNClxmqVs6g77eHqOMBi4O3Qb_LRrYwnVgT


In [3]:
path = "opdrachtgever_2.parquet"
df = get_sharepoint_file(file = path, sub_folder=sub_folder)

✅ Connected with site: Veneficus Data Safe
✅ Found Site ID: veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7

📁 Content of map 'HR Cycle':
   📁 Naam: TypeformData
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData
   📄 Naam: Werknemers_gegevens - Test.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7BD01E5630-2F37-4476-AA36-5BFECEB80C33%7D&file=Werknemers_gegevens%20-%20Test.xlsx&action=default&mobileredirect=true
   📄 Naam: Werknemers_gegevens.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7B144D491D-5701-47A0-81E2-E75F9C99EF74%7D&file=Werknemers_gegevens.xlsx&action=default&mobileredirect=true
DEBUG: Full URL: https://graph.microsoft.com/v1.0/sites/veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7/drive/root:/HR

In [4]:
def combine_newest_files():
    sub_folder = get_secret("TRANSFORMED_RESPONSES_FOLDER")
    items, headers, site_id = get_folder_items(sub_folder)

    df_list = []
    file_names = []
    for item in items:
        item_type = "file" if "file" in item else "map"
        if item_type == "file":
            file_name = item.get("name")
            file_names.append(file_name)
            df = get_sharepoint_file(file_name, sub_folder = sub_folder)
            if df is None:
                print(f"Empty df found: {file_name}")
            else:
                df_list.append(df)

    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        return combined_df, file_names
    else:
        print("No dataframes found to concat.")
    

In [5]:
def move_files(file_names):
    items, headers, site_id = get_folder_items(sub_folder)
    
    for item in items:
            if item['name'] == 'processed' and 'folder' in item:
                processed_folder_id = item['id']
                break
    
    for file_name in file_names:
        file_id = next((i['id'] for i in items if i['name'] == file_name), None)
            
        if file_id:
            move_url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/items/{file_id}"
                
            # De 'parentReference' update verplaatst het bestand
            move_data = {
                    "parentReference": {"id": processed_folder_id},
                    "name": file_name # Je kunt hier eventueel ook de naam aanpassen
            }

    response = requests.patch(move_url, headers=headers, json=move_data)
                
    if response.status_code in [200, 201]:
        print(f"✅ Moved: {file_name}")
    else:
        print(f"❌ Error while moving {file_name}: {response.text}")

In [6]:
combined_df, file_names = combine_newest_files()
for i in range(len(file_names)):
    move_files(file_names)

✅ Connected with site: Veneficus Data Safe
✅ Found Site ID: veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7

📁 Content of map 'HR Cycle/TypeformData/transformed_responses':
   📁 Naam: processed
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData/transformed_responses/processed
   📄 Naam: opdrachtgever_2.parquet
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData/transformed_responses/opdrachtgever_2.parquet
   📄 Naam: opdrachtgever_3.parquet
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData/transformed_responses/opdrachtgever_3.parquet
   📄 Naam: p0lz64nimkjaov7qc6p0lz616l1bthxe
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData/transformed_responses/p0lz64nimkjaov7qc6p0lz616l1bthxe
   📄 N

In [11]:
target_filename="Combined_data_test.xlsx"
upload_and_merge(combined_df, target_filename, combined_df_folder)

✅ Connected with site: Veneficus Data Safe
✅ Found Site ID: veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7

📁 Content of map 'HR Cycle':
   📁 Naam: TypeformData
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData
   📄 Naam: Werknemers_gegevens - Test.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7BD01E5630-2F37-4476-AA36-5BFECEB80C33%7D&file=Werknemers_gegevens%20-%20Test.xlsx&action=default&mobileredirect=true
   📄 Naam: Werknemers_gegevens.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7B144D491D-5701-47A0-81E2-E75F9C99EF74%7D&file=Werknemers_gegevens.xlsx&action=default&mobileredirect=true
DEBUG: Full URL: https://graph.microsoft.com/v1.0/sites/veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7/drive/root:/HR